In [ ]:
import numpy as np

# Función Rastrigin para 2 variables (generalizable a n variables)
def rastrigin(x):
    A = 10
    n = len(x)
    return A * n + np.sum(x**2 - A * np.cos(2 * np.pi * x))

class Ant:
    def __init__(self, num_vars, bounds, alpha=1, beta=1):
        self.position = np.random.uniform(bounds[:, 0], bounds[:, 1], num_vars)
        self.alpha = alpha # peso de la feromona
        self.beta = beta # peso a la carateristica del camino (distancia)

    def choose_next_position(self, pheromones, bounds):
        num_vars = len(bounds)
        probabilities = np.zeros(num_vars)
        
        for i in range(num_vars):
            # cuanto más cerca esté de 0, mayor será el valor heurístico (un empujón interno para ir al centro).
            eta = 1 / (np.abs(self.position[i]) + 1e-10)  # heuristic value refuerzo del camino
            # la probabilidad de moverse en cada eje, elige uno al azar 
            # (pero con preferencia por el que tenga más feromona y mejor heurística), 
            # cambia su valor a una nueva posición aleatoria y se asegura de no salirse de los límites
            probabilities[i] = (pheromones[i] ** self.alpha) * (eta ** self.beta)
        
        probabilities /= np.sum(probabilities)
        
        next_var = np.random.choice(np.arange(num_vars), p=probabilities)
        self.position[next_var] = np.random.uniform(bounds[next_var, 0], bounds[next_var, 1])
        self.position = np.clip(self.position, bounds[:, 0], bounds[:, 1])

def ant_colony_optimization(func, bounds, num_ants=20, max_iter=200, alpha=1, beta=2, evaporation_rate=0.5):
    num_vars = len(bounds)
    bounds = np.array(bounds)
    pheromones = np.ones(num_vars)
    best_position = None
    best_value = float('inf')

    for _ in range(max_iter):
        ants = [Ant(num_vars, bounds, alpha, beta) for _ in range(num_ants)] # Instancia de hormigas
        # En cada iteración, se crean 20 hormigas. 
        # Cada una se mueve, evalúa qué tan buena es su nueva 
        # posición usando la función Rastrigin y, si encuentra 
        # una posición mejor que la mejor registrada en la historia del algoritmo, se guarda.
        for ant in ants:
            ant.choose_next_position(pheromones, bounds)
            current_value = func(ant.position)
            if current_value < best_value:
                best_value = current_value
                best_position = ant.position.copy()
        # Feromonas (pheromones): El rastro físico que dejaron otras hormigas exitosas.
        pheromones *= (1 - evaporation_rate)
        # se reduce a la mitad la evaporación de las feromonas
        for ant in ants:
            # Actualización de feromonas: Después de que todas las hormigas 
            # han evaluado la función, se simula la evaporación de feromonas 
            # (para evitar quedar atrapados en óptimos locales) y se refuerzan 
            # con más feromonas los caminos que condujeron a los mejores valores de la función.
            pheromones += 1 / (1 + func(ant.position)) 
            # Feromona: valor grande si el resultdao de la función es cercano a cero (añadiendo más feromona a ese camino)

    return best_position, best_value

# Parámetros del problema
bounds = [(-5.12, 5.12), (-5.12, 5.12)]  # dos variables (espacio de búsqueda)

# Ejecutar optimización
best_position, best_value = ant_colony_optimization(rastrigin, bounds)

print(f"Mejor posición encontrada: {best_position}")
print(f"Valor de la función Rastrigin: {best_value}")

Mejor posición encontrada: [-0.95139579 -1.02917694]
Valor de la función Rastrigin: 2.5946277839018848
